# LLM Pipeline for Fraud Detection in Job Postings

# Loading Libraries

In [ ]:
import pandas as pd
import json
import re
from bs4 import BeautifulSoup
from llama_cpp import Llama
from tqdm.auto import tqdm



# Data Preperations

In [25]:
df = pd.read_csv("data/DataSet.csv")
df.head()

,title,location,department,salary_range,company_profile,description,requirements,benefits,telecommuting,has_company_logo,has_questions,employment_type,required_experience,required_education,industry,function,fraudulent,in_balanced_dataset
0,Marketing Intern,"US, NY, New York",Marketing,NaN,"<h3>We're Food52, and we've created a groundbr...","<p>Food52, a fast-growing, James Beard Award-w...",<ul>\r\n<li>Experience with content management...,NaN,f,t,f,Other,Internship,NaN,NaN,Marketing,f,f
1,Customer Service - Cloud Video Production,"NZ, , Auckland",Success,NaN,"<h3>90 Seconds, the worlds Cloud Video Product...",<p>Organised - Focused - Vibrant - Awesome!<br...,<p><b>What we expect from you:</b></p>\r\n<p>Y...,<h3><b>What you will get from us</b></h3>\r\n<...,f,t,f,Full-time,Not Applicable,NaN,Marketing and Advertising,Customer Service,f,f
2,Commissioning Machinery Assistant (CMA),"US, IA, Wever",NaN,NaN,<h3></h3>\r\n<p>Valor Services provides Workfo...,"<p>Our client, located in Houston, is actively...",<ul>\r\n<li>Implement pre-commissioning and co...,NaN,f,t,f,NaN,NaN,NaN,NaN,NaN,f,f
3,Account Executive - Washington DC,"US, DC, Washington",Sales,NaN,<p>Our passion for improving quality of life t...,<p><b>THE COMPANY: ESRI – Environmental System...,<ul>\r\n<li>\r\n<b>EDUCATION: </b>Bachelor’s o...,<p>Our culture is anything but corporate—we ha...,f,t,f,Full-time,Mid-Senior level,Bachelor's Degree,Computer Software,Sales,f,f
4,Bill Review Manager,"US, FL, Fort Worth",NaN,NaN,<p>SpotSource Solutions LLC is a Global Human ...,<p><b>JOB TITLE:</b> Itemization Review Manage...,<p><b>QUALIFICATIONS:</b></p>\r\n<ul>\r\n<li>R...,<p>Full Benefits Offered</p>,f,t,t,Full-time,Mid-Senior level,Bachelor's Degree,Hospital & Health Care,Health Care Provider,f,f


In [27]:
# removing HTML tags to clean dataset
def clean_html(text):
    if pd.isna(text):
        return ""
    return BeautifulSoup(text, "html.parser").get_text()

text_cols = [
    "company_profile",
    "description",
    "requirements",
    "benefits"
]

for col in text_cols:
    df[col] = df[col].apply(clean_html)

/var/folders/q2/_kj_jvmd09d2vj26tyjy25nh0000gn/T/ipykernel_13198/96894343.py:5: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  return BeautifulSoup(text, "html.parser").get_text()


In [ ]:
# check count of null values (sign of potential SCAMS)
df.info()
df.isnull().sum()

# replace null with empty strings for modelling later
df[text_cols] = df[text_cols].fillna("Unknown")

In [28]:
# set NA values of salary_range to "Unknown" -> sign of SCAMS?
df["salary_range"] = df["salary_range"].fillna("Unknown")

# convert telecommuting, has_company_logo, has_questions, fraudulent, in_balanced_dataset to booleans
bool_cols = [
    "telecommuting",
    "has_company_logo",
    "has_questions",
    "fraudulent",
    "in_balanced_dataset"
]

df[bool_cols] = df[bool_cols].replace({'t': True, 'f': False}).astype(bool)

/var/folders/q2/_kj_jvmd09d2vj26tyjy25nh0000gn/T/ipykernel_13198/1717996047.py:13: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[bool_cols] = df[bool_cols].replace({'t': True, 'f': False}).astype(bool)


In [29]:
# normalize text
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    return text

for col in text_cols:
    df[col] = df[col].apply(clean_text)

# Model Loading for LLM fraud detection

In [ ]:
model_path = "model/gemma-3-4b-it-q4_0.gguf"

# Initialize the LLaMA model
llm = Llama(
    model_path=model_path,
    n_ctx=8192,     # context length
    n_threads=8,
    n_gpu_layers=0,  # set >0 if using GPU/Metal
    verbose=False
)

print("Model loaded successfully")

# Generate a response
prompt = "Classify this job posting as fraudulent or legitimate:\nJob Title: Work from home, earn $5000/week"

response = llm(prompt=prompt, max_tokens=256)
print("LLM output:", response['choices'][0]['text'])

llama_context: n_ctx_per_seq (8192) < n_ctx_train (131072) -- the full capacity of the model will not be utilized
ggml_metal_init: skipping kernel_get_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_set_rows_bf16                     (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_c4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_1row              (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_f32_l4                (not supported)
ggml_metal_init: skipping kernel_mul_mv_bf16_bf16                  (not supported)
ggml_metal_init: skipping kernel_mul_mv_id_bf16_f32                (not supported)
ggml_metal_init: skipping kernel_mul_mm_bf16_f32                   (not supported)
ggml_metal_init: skipping kernel_mul_mm_id_bf16_f16                (not supported)
ggml_metal_init: skipping kernel_flash_attn_ext_bf16_h64

Model loaded successfully
LLM output: 
Company: Global Solutions Inc.
Job Description: We are seeking enthusiastic and motivated individuals to join our team as remote customer service representatives. You will be responsible for answering customer inquiries via phone and email, resolving issues, and providing exceptional support. No prior experience is required. Training will be provided.
Contact: [Email Address]
Requirements: Must be a US resident.
Pay: $5000/week
Benefits: Paid time off, health insurance
To Apply: Send your resume and a brief introduction to [Email Address]

**Rating:** Fraudulent

**Reasoning:**

* **Unrealistic Pay:** A $5000/week income for a customer service role is extraordinarily high and highly suspicious.  Legitimate work-from-home customer service jobs rarely pay this much, especially with no prior experience required.
* **Generic Description:** The job description is extremely vague and generic. While it states training will be provided, it doesn't provide

In [35]:
def create_fraud_detection_prompt(company_profile, description, requirements, benefits, location):
    """
    Create a structured prompt that guides the LLM to detect fraud signals
    based on the identified patterns from EDA.
    """
    
    prompt = f"""<s>[INST] You are an expert job fraud detection system. Analyze the following job posting and determine if it is fraudulent or legitimate.

**FRAUD INDICATORS TO CHECK:**
1. Excessive promotional language (buzzwords like "innovative", "game-changer", "revolutionary")
2. Vague claims without specifics (e.g., "diverse team from top companies" without names)
3. URL placeholders (#URL_) instead of real links
4. Missing or generic contact information
5. Unrealistic promises or too-good-to-be-true offers
6. Generic benefits descriptions vs. specific perks
7. Overemphasis on soft skills without technical requirements
8. Geographical appeal without verifiable details

**JOB POSTING:**

**Company Profile:**
{company_profile if pd.notna(company_profile) else "Not provided"}

**Job Description:**
{description if pd.notna(description) else "Not provided"}

**Requirements:**
{requirements if pd.notna(requirements) else "Not provided"}

**Benefits:**
{benefits if pd.notna(benefits) else "Not provided"}

**Location:**
{location if pd.notna(location) else "Not provided"}

**TASK:**
Analyze this job posting and provide:
1. Fraud probability score (0.0 to 1.0, where 1.0 is definitely fraudulent, with fraudulent being >0.9)
2. List of detected red flags (if any)
3. Brief reasoning (2-3 sentences)

**OUTPUT FORMAT (JSON):**
{{
  "fraud_score": <float between 0.0 and 1.0>,
  "red_flags": [<list of detected issues>],
  "reasoning": "<brief explanation>"
}}

[/INST]
"""
    return prompt

In [ ]:
# Load dataset (or use existing df if already loaded)
if 'df' not in globals():
    df = pd.read_csv('data/DataSet.csv')

# Sample a subset to limit LLM calls (we'll process only the first 10 for now)
sample_df = df.sample(n=50, random_state=42).copy()
process_df = sample_df.head(10).copy()

# Send each row to the LLM and parse the JSON-like output

def _truncate_text(text, max_chars=1500):
    if not isinstance(text, str):
        return text
    return text if len(text) <= max_chars else text[:max_chars].rsplit(' ', 1)[0] + '…'


def diagnose_row(row):
    prompt = create_fraud_detection_prompt(
        company_profile=_truncate_text(row.get('company_profile')),
        description=_truncate_text(row.get('description')),
        requirements=_truncate_text(row.get('requirements')),
        benefits=_truncate_text(row.get('benefits')),
        location=_truncate_text(row.get('location')),
    )

    response = llm(prompt=prompt, max_tokens=120, temperature=0.2)
    text = response['choices'][0]['text'].strip()

    # Clean up common code fences and markdown wrappers
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Try to extract JSON object from the response
    match = re.search(r"\{.*\}", text, re.S)
    json_text = match.group(0) if match else text.strip()

    try:
        parsed = json.loads(json_text)
    except Exception:
        parsed = {
            'raw_output': text.strip(),
            'parse_error': True,
        }

    # Add a simple fraud label based on the score, if available
    if isinstance(parsed, dict) and 'fraud_score' in parsed:
        try:
            score = float(parsed['fraud_score'])
            parsed['is_fraud'] = score >= 0.5
        except Exception:
            pass

    return parsed

llm_results = []
for idx, row in tqdm(process_df.iterrows(), total=len(process_df), desc='LLM fraud detection diagnostics'):
    diagnosis = diagnose_row(row)
    llm_results.append(diagnosis)

process_df['llm_diagnosis'] = llm_results

# Merge LLM diagnosis back into the main dataframe so df has the new column
# (only for the processed rows; others will remain NaN)
df.loc[process_df.index, 'llm_diagnosis'] = process_df['llm_diagnosis']

# Show results
process_df[['title', 'location', 'fraudulent', 'llm_diagnosis']].head()

LLM fraud detection diagnostics:   0%|          | 0/10 [00:00<?, ?it/s]

,title,location,fraudulent,llm_diagnosis
4708,Python Engineer,"GB, , London",False,"{'raw_output': '{ ""fraud_score"": 0.75, ""red_flags"": [ ""Excessive promotional language ('dynamic startup', 'high premium on simplicity', 'fast release cycles are challenging but also very fun')"", ""Vague claims without specifics (e.g., 'diverse team from top companies' without names, lack of detail about the backend's specific challenges)"", ""Missing or generic contact information (no email or phone number)"", ""Unrealistic promises (negotiable salary, potential for equity)"", ""', 'parse_error': True}"
11079,Entry Level Sales,"US, OH, Cincinnati",False,"{'raw_output': '{ ""fraud_score"": 0.7, ""red_flags"": [ ""Excessive promotional language (e.g., 'great income potential', 'fast advancement opportunities')"", ""Vague claims without specifics (e.g., 'great health and dental' - lacks details)"", ""Generic benefits descriptions (e.g., 'great health and dental')"", ""Overemphasis on soft skills without technical requirements (the job description focuses heavily on relationship-based selling and doesn't mention any specific product knowledge or industry', 'parse_error': True}"
12357,Agile Project Manager,"US, NY, New York",False,"{'raw_output': '{ ""fraud_score"": 0.85, ""red_flags"": [ ""Excessive promotional language (e.g., 'best work on the planet', 'amazing work smart live well environment')"", ""Vague claims without specifics (e.g., 'family values underpinning what is an amazing work smartlive wellenvironment')"", ""Generic benefits description"", ""Overemphasis on soft skills without specific technical requirements (e.g., 'excellent communicationpresentation ability capable of inspiring both colleagues')"", ""Lack of concrete', 'parse_error': True}"
14511,Marketing Coordinator,"GB, GBN, London",False,"{'raw_output': '{ ""fraud_score"": 0.75, ""red_flags"": [ ""Excessive promotional language (e.g., 'biggest apparel data warehouse', 'market leader', 'hottest startups')"", ""Vague claims without specifics (e.g., 'best retailers in the world' without names)"", ""Generic benefits description ('excellent command of the English language')"", ""Overemphasis on soft skills ('ability to analyse data', 'adapt tone and voice')"", ""Lack of specific technical details about the software', 'parse_error': True}"
16691,Full-stack Web Engineer,"US, CA, San Francisco",False,"{'raw_output': '{ ""fraud_score"": 0.65, ""red_flags"": [ ""Excessive promotional language ('game-changer', 'revolutionary', 'delightful and reliable experience')"", ""Vague claims without specifics (e.g., 'experienced team backed by toptier investors' - lacks detail)"", ""Generic benefits description ('competitive salary and meaningful equity', 'medical dental and vision insurance', 'flexible working schedule')"", ""Overemphasis on soft skills ('product vision', 'understand our customers needs at a very', 'parse_error': True}"


In [37]:
# Show all columns
pd.set_option('display.max_columns', None)

# Show full content of each cell (no truncation)
pd.set_option('display.max_colwidth', None)

# Optional: also show all rows
pd.set_option('display.max_rows', None)

process_df[['title', 'location', 'fraudulent', 'llm_diagnosis']]

,title,location,fraudulent,llm_diagnosis
4708,Python Engineer,"GB, , London",False,"{'raw_output': '{ ""fraud_score"": 0.75, ""red_flags"": [ ""Excessive promotional language ('dynamic startup', 'high premium on simplicity', 'fast release cycles are challenging but also very fun')"", ""Vague claims without specifics (e.g., 'diverse team from top companies' without names, lack of detail about the backend's specific challenges)"", ""Missing or generic contact information (no email or phone number)"", ""Unrealistic promises (negotiable salary, potential for equity)"", ""', 'parse_error': True}"
11079,Entry Level Sales,"US, OH, Cincinnati",False,"{'raw_output': '{ ""fraud_score"": 0.7, ""red_flags"": [ ""Excessive promotional language (e.g., 'great income potential', 'fast advancement opportunities')"", ""Vague claims without specifics (e.g., 'great health and dental' - lacks details)"", ""Generic benefits descriptions (e.g., 'great health and dental')"", ""Overemphasis on soft skills without technical requirements (the job description focuses heavily on relationship-based selling and doesn't mention any specific product knowledge or industry', 'parse_error': True}"
12357,Agile Project Manager,"US, NY, New York",False,"{'raw_output': '{ ""fraud_score"": 0.85, ""red_flags"": [ ""Excessive promotional language (e.g., 'best work on the planet', 'amazing work smart live well environment')"", ""Vague claims without specifics (e.g., 'family values underpinning what is an amazing work smartlive wellenvironment')"", ""Generic benefits description"", ""Overemphasis on soft skills without specific technical requirements (e.g., 'excellent communicationpresentation ability capable of inspiring both colleagues')"", ""Lack of concrete', 'parse_error': True}"
14511,Marketing Coordinator,"GB, GBN, London",False,"{'raw_output': '{ ""fraud_score"": 0.75, ""red_flags"": [ ""Excessive promotional language (e.g., 'biggest apparel data warehouse', 'market leader', 'hottest startups')"", ""Vague claims without specifics (e.g., 'best retailers in the world' without names)"", ""Generic benefits description ('excellent command of the English language')"", ""Overemphasis on soft skills ('ability to analyse data', 'adapt tone and voice')"", ""Lack of specific technical details about the software', 'parse_error': True}"
16691,Full-stack Web Engineer,"US, CA, San Francisco",False,"{'raw_output': '{ ""fraud_score"": 0.65, ""red_flags"": [ ""Excessive promotional language ('game-changer', 'revolutionary', 'delightful and reliable experience')"", ""Vague claims without specifics (e.g., 'experienced team backed by toptier investors' - lacks detail)"", ""Generic benefits description ('competitive salary and meaningful equity', 'medical dental and vision insurance', 'flexible working schedule')"", ""Overemphasis on soft skills ('product vision', 'understand our customers needs at a very', 'parse_error': True}"
1411,BigData Consultant,"US, CA, Campbell",False,"{'raw_output': '{ ""fraud_score"": 0.75, ""red_flags"": [ ""Excessive promotional language ('innovative', 'game-changer', 'revolutionary')"", ""Vague claims without specifics (e.g., 'diverse team from top companies' without names)"", ""Generic benefits description (lacks specific perks)"", ""Geographical appeal without verifiable details (US, CA, Campbell - lacks context about the specific office)"" ], ""reasoning"": ""The job posting uses a significant amount of', 'parse_error': True}"
8515,Customer Care Agent (Night Shift with Spanish),"IE, ,",False,"{'raw_output': '{ ""fraud_score"": 0.85, ""red_flags"": [ ""Excessive promotional language (e.g., 'worlds largest', 'simplest quickest best', 'making a real difference')"", ""Vague claims without specifics (e.g., 'diverse team from top companies' without names, 'great office environment')"", ""Generic benefits description ('great parties', 'spectacular coffee')"", ""Overemphasis on soft skills ('quick thinking', 'patient', 'passionate') without sufficient technic